In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("TensorFlow version:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [2]:
%pip install tensorflow

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("TensorFlow version:", tf.__version__)

Pandas version: 3.0.5
NumPy version: 2.3.5
TensorFlow version: 2.21.0


In [3]:
from pathlib import Path

DATA_PATH = Path("../data/Tweets.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df[["text", "airline_sentiment"]].head())

Dataset shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']
                                                text airline_sentiment
0                @VirginAmerica What @dhepburn said.           neutral
1  @VirginAmerica plus you've added commercials t...          positive
2  @VirginAmerica I didn't today... Must mean I n...           neutral
3  @VirginAmerica it's really aggressive to blast...          negative
4  @VirginAmerica and it's a really big bad thing...          negative


In [4]:
import re
import html

def clean_tweet(text):
    text = html.unescape(str(text))
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_tweet)

print("Missing text:", df["clean_text"].isna().sum())
print("\nSentiment distribution:")
print(df["airline_sentiment"].value_counts())

print("\nCleaned examples:")
print(df[["clean_text", "airline_sentiment"]].head())

Missing text: 0

Sentiment distribution:
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Cleaned examples:
                                          clean_text airline_sentiment
0                                         What said.           neutral
1  plus you've added commercials to the experienc...          positive
2  I didn't today... Must mean I need to take ano...           neutral
3  it's really aggressive to blast obnoxious "ent...          negative
4           and it's a really big bad thing about it          negative


In [5]:

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = df["clean_text"].astype(str)
y = df["airline_sentiment"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq, maxlen=50, padding="post", truncating="post"
)
X_test_pad = pad_sequences(
    X_test_seq, maxlen=50, padding="post", truncating="post"
)

print("Training data:", X_train_pad.shape)
print("Testing data:", X_test_pad.shape)
print("Vocabulary size:", len(tokenizer.word_index))
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)


Training data: (11712, 50)
Testing data: (2928, 50)
Vocabulary size: 12193
Training labels: (11712,)
Testing labels: (2928,)


NameError: name 'y_train' is not defined

In [1]:

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# it convert labels into numbers 
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Label order:", label_encoder.classes_)

# LSTM neural network
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, mask_zero=True),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax")
])

#  model to trauning k lie configure kiya 
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# structure of model
model.summary()


NameError: name 'y_train' is not defined

In [2]:
print("Label order:", label_encoder.classes_)

AttributeError: 'LabelEncoder' object has no attribute 'classes_'

In [7]:

model.build(input_shape=(None, 50))

#  model ki details
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 675,203 (2.58 MB)

 Trainable params: 675,203 (2.58 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train_encoded,
    validation_split=0.1,
    epochs=8,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/8
165/165 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - accuracy: 0.6796 - loss: 0.7329 - val_accuracy: 0.7363 - val_loss: 0.6162
Epoch 2/8
165/165 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.8063 - loss: 0.4802 - val_accuracy: 0.7978 - val_loss: 0.5539
Epoch 3/8
165/165 ━━━━━━━━━━━━━━━━━━━━ 13s 79ms/step - accuracy: 0.8836 - loss: 0.3212 - val_accuracy: 0.7892 - val_loss: 0.5751
Epoch 4/8
165/165 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.9197 - loss: 0.2307 - val_accuracy: 0.7790 - val_loss: 0.6418


In [9]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test_encoded,
    verbose=1
)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7941 - loss: 0.5443
Test Accuracy: 79.41%
Test Loss: 0.5443


In [10]:

from sklearn.metrics import classification_report, confusion_matrix

# Test tweets predictions hhihihihi
y_pred_prob = model.predict(X_test_pad, verbose=0)
y_pred_encoded = np.argmax(y_pred_prob, axis=1)

# Numbers are convertedd into sentiment 
y_pred = label_encoder.inverse_transform(y_pred_encoded)

#  performance report
print(classification_report(
    y_test,
    y_pred,
    labels=["negative", "neutral", "positive"],
    zero_division=0
))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred,
    labels=["negative", "neutral", "positive"]
))


              precision    recall  f1-score   support

    negative       0.83      0.93      0.88      1835
     neutral       0.67      0.52      0.58       620
    positive       0.76      0.62      0.69       473

    accuracy                           0.79      2928
   macro avg       0.75      0.69      0.72      2928
weighted avg       0.78      0.79      0.78      2928

Confusion Matrix:
[[1708   89   38]
 [ 243  322   55]
 [ 107   71  295]]


In [12]:

from pathlib import Path
import joblib

# path 
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)


model.save(models_dir / "lstm_sentiment_model.keras")

joblib.dump(tokenizer, models_dir / "lstm_tokenizer.pkl")
joblib.dump(label_encoder, models_dir / "lstm_label_encoder.pkl")

print("LSTM model and supporting files are save")


LSTM model and supporting files are save


In [13]:

for file in sorted(models_dir.iterdir()):
    print(file.name)


logistic_regression.pkl
lstm_label_encoder.pkl
lstm_sentiment_model.keras
lstm_tokenizer.pkl
tfidf_vectorizer.pkl


In [14]:

from pathlib import Path
import joblib
from tensorflow.keras.models import load_model

models_dir = Path("../models")

# Save file loads here
loaded_model = load_model(
    models_dir / "lstm_sentiment_model.keras"
)
loaded_tokenizer = joblib.load(
    models_dir / "lstm_tokenizer.pkl"
)
loaded_encoder = joblib.load(
    models_dir / "lstm_label_encoder.pkl"
)

print("Model loaded:", type(loaded_model).__name__)
print("Classes:", loaded_encoder.classes_)
print("Tokenizer loaded:", len(loaded_tokenizer.word_index))


Model loaded: Sequential
Classes: ['negative' 'neutral' 'positive']
Tokenizer loaded: 12193


In [15]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

test_tweets = [
    "I love this airline, the service was amazing",
    "My flight was okay, nothing special.",
    "My flight was terrible and the service was awful."
]

# Text to number convertion 
sequences = loaded_tokenizer.texts_to_sequences(test_tweets)

# each token has 50 length
padded = pad_sequences(
    sequences,
    maxlen=50,
    padding="post",
    truncating="post"
)

# Sentiment predictions
probabilities = loaded_model.predict(padded, verbose=0)
predicted_ids = probabilities.argmax(axis=1)
predicted_labels = loaded_encoder.inverse_transform(predicted_ids)

for tweet, sentiment in zip(test_tweets, predicted_labels):
    print(f"Tweet: {tweet}")
    print(f"Predicted sentiment: {sentiment}\n")


Tweet: I love this airline, the service was amazing
Predicted sentiment: positive

Tweet: My flight was okay, nothing special.
Predicted sentiment: negative

Tweet: My flight was terrible and the service was awful.
Predicted sentiment: negative



In [16]:

import joblib
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

models_dir = Path("../models")

# yha se Logistic Regression model
lr_model = joblib.load(models_dir / "logistic_regression.pkl")
tfidf = joblib.load(models_dir / "tfidf_vectorizer.pkl")

# Saved LSTM files load 
lstm_model = load_model(models_dir / "lstm_sentiment_model.keras")
tokenizer = joblib.load(models_dir / "lstm_tokenizer.pkl")
label_encoder = joblib.load(models_dir / "lstm_label_encoder.pkl")


tweets = [
    "I love this airline, the service was amazing!",
    "My flight was okay, nothing special.",
    "My flight was terrible and the service was awful."
]

#  Logistic Regression hui hui bado badi hehe
X_tfidf = tfidf.transform(tweets)
lr_predictions = lr_model.predict(X_tfidf)

# LSTM 
sequences = tokenizer.texts_to_sequences(tweets)
padded = pad_sequences(
    sequences, maxlen=50, padding="post", truncating="post"
)
probabilities = lstm_model.predict(padded, verbose=0)
lstm_ids = probabilities.argmax(axis=1)
lstm_predictions = label_encoder.inverse_transform(lstm_ids)

# result
results = pd.DataFrame({
    "Tweet": tweets,
    "Logistic Regression": lr_predictions,
    "LSTM": lstm_predictions
})

display(results)


,Tweet,Logistic Regression,LSTM
0,"I love this airline, the service was amazing!",positive,positive
1,"My flight was okay, nothing special.",negative,negative
2,My flight was terrible and the service was awful.,negative,negative


In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Label order:", label_encoder.classes_)

NameError: name 'y_train' is not defined

In [4]:
print("y_train exists:", "y_train" in globals())
print("y_test exists:", "y_test" in globals())

y_train exists: False
y_test exists: False


In [6]:
print(df.shape)
print(df.columns.tolist())

NameError: name 'df' is not defined

In [7]:
print(df.shape)
print(df.columns.tolist())

NameError: name 'df' is not defined

In [9]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "Tweets.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully!
Shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


In [10]:
print("clean_text exists:", "clean_text" in df.columns)
print("airline_sentiment exists:", "airline_sentiment" in df.columns)

clean_text exists: False
airline_sentiment exists: True


In [11]:
print("text column exists:", "text" in df.columns)
print(df[["text", "airline_sentiment"]].head())

text column exists: True
                                                text airline_sentiment
0                @VirginAmerica What @dhepburn said.           neutral
1  @VirginAmerica plus you've added commercials t...          positive
2  @VirginAmerica I didn't today... Must mean I n...           neutral
3  @VirginAmerica it's really aggressive to blast...          negative
4  @VirginAmerica and it's a really big bad thing...          negative


In [12]:
import re
import html

def clean_tweet(text):
    text = html.unescape(str(text))
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_tweet)

print(df[["clean_text", "airline_sentiment"]].head())

                                          clean_text airline_sentiment
0                                         What said.           neutral
1  plus you've added commercials to the experienc...          positive
2  I didn't today... Must mean I need to take ano...           neutral
3  it's really aggressive to blast obnoxious "ent...          negative
4           and it's a really big bad thing about it          negative


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df["clean_text"].astype(str)
y = df["airline_sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Label order:", label_encoder.classes_)

Label order: ['negative' 'neutral' 'positive']
